In [ ]:
import sys

# Install dependencies
if not 'pandas' in sys.modules:
    !pip install pandas
if not 'plotly' in sys.modules:
    !pip install plotly
if not 'kaleido' in sys.modules:
    !pip install -U kaleido
if not 'ipykernel' in sys.modules:
    !pip install ipykernel
# !pip install --upgrade nbformat


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import visualization_tools as vt
import toolbox as tb
import plotly.graph_objects as go
import os

sns.set_theme()

ORIENTED = False
# ORIENTED = True

In [ ]:
def split_merge_data_type(df):
    dfs_to_add = []
    indices_to_drop = []

    for data_type in df["data"].unique():
        if "merge" in data_type:
            df_all = df[df["data"] == data_type]
            
            df_no_noise = df_all[(df_all["Noise position"] == 0) & (df_all["Noise normal"] == 0)].copy()
            df_noise = df_all[(df_all["Noise position"] != 0) | (df_all["Noise normal"] != 0)].copy()
            df_noiseposition = df_all[(df_all["Noise position"] != 0) & (df_all["Noise normal"] == 0)].copy()
            df_noisenormal = df_all[(df_all["Noise position"] == 0) & (df_all["Noise normal"] != 0)].copy()

            df_no_noise["data"] = df_no_noise["data"].str.replace("merge", "nonoise")
            df_noise["data"] = df_noise["data"].str.replace("merge", "noise")
            df_noiseposition["data"] = df_noiseposition["data"].str.replace("merge", "noiseposition")
            df_noisenormal["data"] = df_noisenormal["data"].str.replace("merge", "noisenormal")

            indices_to_drop.extend(df_all.index.tolist())
            
            dfs_to_add.extend([df_no_noise, df_noise, df_noiseposition, df_noisenormal])

        if ("outlier" in data_type) or ("flip" in data_type):
            df_all = df[df["data"] == data_type]
            df_no_noise = df_all[(df_all["Noise position"] == 0) & (df_all["Noise normal"] == 0)].copy()
            
            indices_to_drop.extend(df_all.index.tolist())
            dfs_to_add.append(df_no_noise)

    if indices_to_drop:
        df = df.drop(index=indices_to_drop)
    
    if dfs_to_add:
        dfs_to_add.insert(0, df)
        df = pd.concat(dfs_to_add, ignore_index=True)

    return df

def split_nonoise_noise(df):
        df_all = df.copy()
        
        df_no_noise = df_all[df_all["data"].str.contains("nonoise")]
        df_noise = df_all[~df_all["data"].str.contains("nonoise")]
    
        return df_no_noise, df_noise

In [ ]:
diagramDir = "diagrams/"

if not os.path.exists(diagramDir):
    os.makedirs(diagramDir)

In [ ]:
statsDir_implicit = "../results/implicit/stats/"
statsDir_implicit_helios = "../results/implicit_helios/stats/"
statsDir_implicit_outlier = "../results/implicit_outlier/stats/"
statsDir_implicit_flip = "../results/implicit_flip/stats/"
statsDir_implicit_float = "../results/implicit_float/stats/"

In [ ]:
def remove_problematic_values(df):
    error_columns = [ 
        "Timings (mean)",
    ]
    for col in error_columns:
        if col in df.columns:
            df.loc[df[col] <= 0, col] = np.nan

    return df

In [ ]:

data_list = []

data_list.append(tb.open_data_all(statsDir_implicit, added_param=[["data", "merge"], ["dataset", "Implicit"], ["type", "double"]], recursive=False))
data_list.append(tb.open_data_all(statsDir_implicit_float, added_param=[["data", "merge"], ["dataset", "Implicit"], ["type", "float"]], recursive=False))
data_list.append(tb.open_data_all(statsDir_implicit_helios, added_param=[["data", "merge helios"], ["dataset", "Implicit"], ["type", "double"]], recursive=False))
data_list.append(tb.open_data_all(statsDir_implicit_flip, added_param=[["data", "flip"], ["dataset", "Implicit"], ["type", "double"]], recursive=False))
data_list.append(tb.open_data_all(statsDir_implicit_outlier, added_param=[["data", "outlier"], ["dataset", "Implicit"], ["type", "double"]], recursive=False))
data_list = pd.concat(data_list, ignore_index=True)
print (data_list["Shape"].unique())

data_new = data_list.copy()
data_list = [ data_new ]

data_all = pd.concat(data_list, ignore_index=True)

data_all = tb.clean_data(data_all)
data_all = remove_problematic_values(data_all)

print(data_all["Method"].unique())


# rename all methods named as XXX_ponca to only XXX
data_all["Method"] = data_all["Method"].str.replace("_PONCA", "")

# Timings (mean) from nanosecond to millisecond
data_all["Timings (mean)"] = data_all["Timings (mean)"] / 1e6
data_all = split_merge_data_type(data_all)

for i, noise_p in enumerate (data_all["Noise position"].unique()) :
    data_all.loc[data_all["Noise position"]==noise_p, "Noise position class"] = i
for j, noise_n in enumerate(data_all["Noise normal"].unique()) :
    data_all.loc[data_all["Noise normal"]==noise_n, "Noise normal class"] = j

for k in range(len(data_all["Noise position"].unique())) :
    data_all.loc[(data_all["Noise position class"]==k) & (data_all["Noise normal class"]==k), "Noise class"] = k

data_all_methods = data_all.copy()

all_methods, methods = tb.methods(data_all, ORIENTED)
# all_methods.extend(data_all["Method"].unique().tolist())
# methods.extend(data_all["Method"].unique().tolist())

data_all = data_all[data_all["Method"].isin(all_methods)]

data_nonoise, data_noise = split_nonoise_noise(data_all)

color_map_method = {}
for method in all_methods:
    color_map_method[method] = plt.cm.get_cmap("tab20")(all_methods.index(method))

color_map_to_value = {} 
color_map_to_value_soft = {} 
for method in color_map_method.keys() :
    color_map_to_value[method] = "rgba(" + str(int(color_map_method[method][0]*255)) + ", " + str(int(color_map_method[method][1]*255)) + ", " + str(int(color_map_method[method][2]*255)) + ", 1)"
    color_map_to_value_soft[method] = "rgba(" + str(int(color_map_method[method][0]*255)) + ", " + str(int(color_map_method[method][1]*255)) + ", " + str(int(color_map_method[method][2]*255)) + ", 0.2)"

In [ ]:
properties = {
    "Mean curvature (mean)": tb.mean_curvature_estimation,
    "K1 mean": tb.principal_curvature_estimation,
    "K2 mean": tb.principal_curvature_estimation,
    "Timings (mean)": all_methods,
    "D1 mean": tb.curvature_direction,
    "D2 mean": tb.curvature_direction,
    "normal mean": tb.normal_estimation,
    "iShape mean": tb.principal_curvature_estimation
}

In [ ]:
title = "All"

data_implicits = data_all[data_all["dataset"] == "Implicit"]
data_implicits = data_implicits[data_implicits["type"] == "double"]
print (data_implicits["data"].unique())

vt.base_fonts['line_width'] = 1

cols = [
    ("Method", properties["Mean curvature (mean)"], "Mean curvature (mean)", "RMSE Mean curvature (log)", "log"),
    ("Method", properties["D1 mean"], "D1 mean", "RMSAE Min direction (log)", "log"),    
    ("Method", properties["normal mean"], "normal mean", "RMSAE Normal"),
]
rows = [
    ( "data", ["nonoise"], "Radius", "Radius NN" ),
    ( "data", ["noiseposition"], "Radius", "Noise position" ),
    ( "data", ["noisenormal"], "Radius", "Noise normal" ),
    ( "data", ["flip"], "Radius", "Radius F" ),
    ( "data", ["outlier"], "Radius", "Radius O" ),
    ( "data", ["nonoise helios"], "Radius", "Radius NN-H" ),
]

y_labels = ["No noise", "Noise position", "Noise normal", "Normal flips", "Outliers", "DGtal + Helios"]

base_fonts = {
    'title_font': 18,
    'axis_title_font': 13,
    'axis_tick_font': 10,
    'annotation_font': 12,
    'legend_font' : 12,
    'font_factor': 1.35,
    'font_family': 'Times New Roman',
}


to_legend = "Method"

subplot_data = vt.create_subplots(data_implicits, rows, cols, to_legend, color_map_to_value)

fig = vt.create_modular_subplots(subplot_data, title=title, fonts=base_fonts, page_width=718.75, graph_ratio=0.43, show_titles=False, collapse_x=False, collapse_y=False, show_legend=True, row_ylabels=y_labels)
fig.show()
fig.write_image(diagramDir + title + ".png", width=fig.layout.width, height=fig.layout.height)
fig.write_image(diagramDir + title + ".pdf", width=fig.layout.width, height=fig.layout.height)

In [ ]:
title = "Timings"

data_implicits = data_all[data_all["dataset"] == "Implicit"]
# Only noise position and noise normal
data_implicits = data_implicits[(data_implicits["data"].str.contains("nonoise")) | (data_implicits["data"].str.contains("noiseposition")) | (data_implicits["data"].str.contains("noisenormal"))]

vt.base_fonts['line_width'] = 1

cols = [
    ("Method", all_methods, "Timings (mean)", "Timings (ms)", "log"),
    ("Method", properties["Mean curvature (mean)"], "Mean curvature (mean)", "RMSE Mean curvature (log)", "log"),
    ("Method", properties["normal mean"], "normal mean", "RMSAE normal"),
]
rows = [
    ( "type", ["double"], "Radius", "Radius" ),
    ( "type", ["float"], "Radius", "Radius" ),
]

y_labels = ["double", "float"]
to_legend = "Method"

subplot_data = vt.create_subplots(data_implicits, rows, cols, to_legend, color_map_to_value)

fig = vt.create_modular_subplots(subplot_data, title=title, fonts=base_fonts, graph_ratio=0.55, page_width=(718.75), show_titles=False, collapse_x=False, collapse_y=False, row_ylabels=y_labels)
fig.show()
fig.write_image(diagramDir + title + ".png", width=fig.layout.width, height=fig.layout.height)
fig.write_image(diagramDir + title + ".pdf", width=fig.layout.width, height=fig.layout.height)